# start

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from arepy.read_write import binary_read as rsnap
from arepy.read_write import binary_write as wsnap
from arepy.utility import cgs_constants as cgs
from arepy.utility import snap_utility as sutil
import matplotlib.colors as mc
from scipy.interpolate import NearestNDInterpolator
from numpy import uint32, uint64, float64, float32
import math
from scipy import signal
from scipy import ndimage
from copy import copy, deepcopy
from scipy.spatial import Delaunay, Voronoi
from collections import defaultdict
import struct
import sys
import time
import bisect
import random
from numpy import uint16, uint32, uint64, float64, float32
from sf3dmodels.grid import fillgrid
from sf3dmodels.arepo import UniqueCells
from sf3dmodels.utils import (constants as sfc, units as sfu)
from sf3dmodels.Model import Struct
import h5py
def round_label(n):
    # Round down using math.floor.
    if n - math.floor(n) == 0.5:
        return math.floor(n)
    else:
        return round(n)

In [ ]:

######################################################################################################
input_file = '../origin/higher_res/snap_256.hdf5'
di = 5.5
rotate = 65
dil = round_label(di)
label = f'offarm{dil}{dil+1}d{rotate}'
output_file = f'./voronoi_grid_{label}.dat'

theta=360-rotate
xmin=500 + di*10
xmax=500 + di*10 + 10
ymin=490 + 5
ymax=510 - 5
zmin=490 + 5
zmax=510 - 5
print(xmin,xmax,ymin,ymax,zmin,zmax,label,output_file)
#*****************************************************************************************************
limit = (1500/2)/100 * 1.00001 #1.00001*max_off ()is the half length of the generate grid box 500pc->360 300pc -> 250 200pc->150
l_max = 2*limit
veloffx, veloffy, veloffz = [0., 0., 0] # Reference offsets for x, y, z.
#######################################################################################################


In [ ]:

#internal arepo units in cgs
#needs possibly to be changed for different simulations
arepoLength = 100*sfu.pc #100 * 3.0856e16 #m  （arepoLength is 100pc)
arepoMass = 1.991e30 #kg #1.991e33 for g
arepoVel = 1.0e3 #m s-1 #1.0e5 for cm s-1
arepoVol = (arepoLength)**3 # m3

arepoDensity = arepoMass/arepoVol #kg m-3
arepoEnergy= arepoMass*arepoVel*arepoVel # kg m2 s-2

grid_id = 50           #polaris grid ID (50 = voronoi)
data_ids = [0, 2, 3, 7, 8, 9, 17, 18] #polaris header #[0, 2, 3, 4, 5, 6, 7, 8, 9, 17, 17, 17] #polaris header

#constants in SI
mp = 1.6726231e-27   # Proton mass.
mco_rat = 28.0101 / 1.00794   # Hydrogen: 1.00794 g/mol; CO: 28.0101 g/mol.
mHI = 1
CLR_LINE =   "                                                               \r"

#arepo header
header_names = ('num_particles', 'mass', 'time', 'redshift', 'flag_sfr', 'flag_feedback', 'num_particles_total', 'flag_cooling', \
                'num_files', 'boxsize', 'omega0', 'omegaLambda', 'hubble0', 'flag_stellarage', 'flag_metals', \
                'npartTotalHighWord', 'flag_entropy_instead_u', 'flag_doubleprecision', \
                'flag_lpt_ics', 'lpt_scalingfactor', 'flag_tracer_field', 'composition_vector_length', 'buffer')

header_sizes = ((uint32, 6), (float64, 6), (float64, 1), (float64, 1), (uint32, 1), (uint32, 1), \
                (uint32, 6), (uint32, 1), (uint32, 1), (float64, 1), (float64, 1), (float64, 1), \
                (float64, 1), (uint32, 1), (uint32, 1), (uint32, 6), (uint32, 1), (uint32, 1), \
                (uint32, 1), (float32, 1), (uint32, 1), (uint32, 1), (np.uint8, 40))

def is_in_hull(a, x):
    i = bisect.bisect_left(a, x)   # Find the left insertion index of x in sorted array a.
    if i != len(a) and a[i] == x:   # Check that i is within the array and a[i] equals x.
        return -1

    return 1

def find_neighbors(tri):  # tri is a Delaunay triangulation.
    neighbors = defaultdict(set)  # Initialize the dictionary.

    tri_len=len(tri.simplices) # Each row of tri.simplices contains the vertices of a Delaunay simplex.
    counter = 0

    for simplex in tri.simplices: # A simplex.
        if counter % 100000 == 0: # Update progress every 100000 simplices.
            sys.stdout.write('Determining neighbors: ' + str(100.0 * counter / tri_len) + '%    \r')   # Display progress.
            sys.stdout.flush()   # Flush the output buffer after printing.

        counter+=1

        for idx in simplex: # idx is a vertex of the current simplex.
            other = set(simplex)  # Create a set of the simplex vertices.
            other.remove(idx)   # Remove the current vertex.
            neighbors[idx] = neighbors[idx].union(other)   # Add connected vertices to the neighbours dictionary.

    return neighbors

def create_convex_hull_list(hull):
    neighbors = defaultdict(set)

    for simplex in hull:
        for idx in simplex:
            other = set(simplex)
            other.remove(idx)
            neighbors[0] = neighbors[0].union(other)   

    return sorted(list(neighbors[0]))        # Return the sorted list of convex-hull vertices.


def tetrahedron_volume(a, b, c, d):     # Compute tetrahedron volumes.
    return np.abs(np.einsum('ij,ij->i', a-d, np.cross(b-d, c-d))) / 6           # Use d as the origin: V = |(A cross B) dot C| / 6, with A = a-d, etc.


from scipy.spatial.distance import pdist

def tetrahedron_volume_and_distances(a, b, c, d):
    # a, b, c, d have shape (n, 3), where n is the number of tetrahedra.
    #volumes = np.abs(np.einsum('ij,ij->i', a-d, np.cross(b-d, c-d))) / 6
    max_distances = []
    avg_distances = []

    for i in range(len(a)):
        # Build a 2-D array containing the four vertices of each tetrahedron.
        points = np.array([a[i], b[i], c[i], d[i]])
        # Compute all pairwise distances between vertices.
        distances = pdist(points, 'euclidean')
        # Compute the maximum and mean distances.
        max_distances.append(np.max(distances))
        avg_distances.append(np.mean(distances))
    
    # Return the averages of the maximum and mean vertex distances.
    #total_volume = np.sum(volumes)
    avg_max_distance = np.mean(max_distances)
    avg_avg_distance = np.mean(avg_distances)
    
    return  avg_max_distance, avg_avg_distance


In [ ]:

#******************************
#AREPO 3D-grids (not projected)
# See the data-output table at the URL below for field definitions.
# https://arepo-code.org/wp-content/userguide/snapshotformat.html
#******************************
rsnap.io_flags['variable_metallicity']=False
rsnap.io_flags['time_steps']=True
rsnap.io_flags['mc_tracer']=False
rsnap.io_flags['sgchem']=True
#rsnap.io_flags['MHD']=True #There is no magnetic field information


#******************************
#READING SNAPSHOT
#******************************
#0 <KeysViewHDF5 ['ChemicalAbundances', 'Coordinates', 'Density', 'DustTemperature', 'InternalEnergy', 'Masses', 'ParticleIDs', 'PotentialPeak', 'Velocities']>



#####################################
###  hdf5 file
#####################################
h5_o=h5py.File(input_file,'r')
h5=h5_o['PartType0']


d_gas = {'pos': h5['Coordinates'][:],
         'vel': h5['Velocities'][:],
         'id': h5['ParticleIDs'][:],
         'mass': h5['Masses'][:],
         'u_therm': h5['InternalEnergy'][:] ,
         'rho': h5['Density'][:] ,
         'chem': h5['ChemicalAbundances'][:],
         'tdust': h5['DustTemperature'][:]
        }



data = {
        }




data_o=deepcopy(d_gas)




#******************************************
#cut region
#******************************************

#0 <KeysViewHDF5 ['ChemicalAbundances', 'Coordinates', 'Density', 'DustTemperature', 'InternalEnergy', 'Masses', 'ParticleIDs', 'PotentialPeak', 'Velocities']>

gasx_otheta = data_o['pos'][:,0] - 500
gasy_otheta = data_o['pos'][:,1] - 500
  
data_o['pos'][:,0] = gasx_otheta*np.cos(theta*np.pi/180) - gasy_otheta*np.sin(theta*np.pi/180) + 500 
data_o['pos'][:,1] = gasy_otheta*np.cos(theta*np.pi/180) + gasx_otheta*np.sin(theta*np.pi/180) + 500

pos_o= data_o['pos']
mass_o= data_o['mass']

gasx_o = pos_o[:,0]
gasy_o = pos_o[:,1] 
gasz_o = pos_o[:,2] 



a=np.where((gasx_o < xmax) & (gasx_o > xmin))[0]
a=a.tolist()
b=np.where((gasy_o < ymax) & (gasy_o > ymin))[0]
b=b.tolist()
az=np.where((gasz_o < zmax) & (gasz_o > zmin))[0]
az=az.tolist()

extract_area_gas=list(set(a).intersection(set(b)).intersection(set(az)))

extract_area=extract_area_gas
print('the number of gas cell:',len(extract_area_gas),len(extract_area))
n_gas_extract = len(extract_area_gas)

total_extract = len(extract_area)


data['pos']=data_o['pos'][extract_area]
data['vel']=data_o['vel'][extract_area]
data['id']=data_o['id'][extract_area]
data['mass']=data_o['mass'][extract_area]


data['u_therm']=data_o['u_therm'][extract_area_gas]
data['rho']=data_o['rho'][extract_area_gas]
data['chem']=data_o['chem'][extract_area_gas]
data['tdust']=data_o['tdust'][extract_area_gas]


header = {}

header['num_particles']=data['id'][:].shape
header_o=deepcopy(header)

xHe = 0.1 #Helium abundance


yn = np.float64(data['rho'])*arepoDensity/ ((1. + 4.0 * xHe) * mp) #Number density of H-species 
energy = np.float64(data['u_therm'] * data['rho']) * arepoEnergy/arepoVol
yntot = (1. + xHe - data['chem'][:,0] + data['chem'][:,1]) * yn
data['tgas'] = 2.*energy / (3.*yntot*sfc.kb)



#*************************************
#DICTS TO OBJECTS AND SOME DEFINITIONS
#*************************************
h_obj = Struct(**header)    
d_obj = Struct(**data)


#******************************
#PHYSICAL PROPERTIES
#******************************
pos = d_obj.pos[:] 
gasmass = data['mass']


print('Mass in gas (AREPO)', np.sum(gasmass))




In [ ]:
def get_sound_speed(T):
    return np.sqrt(5/3. * sfc.kb * T / sfc.mH)     #

#*******************************************
#PRE-PROCESSING FOR POLARIS
#*******************************************
#Function to prepare the physical data:
def get_prop_dicts(id_pos):

    dens_3d = d_obj.rho[id_pos] * arepoDensity / ((1. + 4.0 * xHe) * mp) #kg/m3 --> Number density of H nucleons 
    chem_H2 = d_obj.chem[:,0][id_pos]
    chem_Hplus = d_obj.chem[:,1][id_pos]
    chem_CO = d_obj.chem[:,2][id_pos]
    dens_H2 = dens_3d*chem_H2
    dens_Hplus = dens_3d*chem_Hplus
    coords = ['x','y','z']
    mean_vel = [np.mean(d_obj.vel[:,i][id_pos]) for i in range(3)]
    vel_3d = Struct( **{coords[i]: (d_obj.vel[:,i][id_pos] - mean_vel[i]) * arepoVel for i in range(3)} )
    chem_HI = 1 - chem_Hplus -2* chem_H2
    prop = {'dens_Hnuc': dens_3d,
            'chem_H2': chem_H2,
            'chem_Hplus': chem_Hplus,
            'chem_HI': chem_HI ,
            'dens_H': dens_3d - 2*dens_H2 - dens_Hplus,
            'temp_gas': d_obj.tgas[id_pos],
            'temp_dust': d_obj.tdust[id_pos],
            'doppler': get_sound_speed(d_obj.tgas[id_pos]),
            'chem_CO': d_obj.chem[:,2][id_pos],
            #'chem_CO': d_obj.chem[:,2][id_pos]/69., #/69 stands for the N(12CO)/N(13CO) ratio
            'vel_x': vel_3d.x + veloffx*arepoVel,
            'vel_y': vel_3d.y + veloffy*arepoVel,
            'vel_z': vel_3d.z + veloffz*arepoVel,
            'gtdratio': 100 * np.ones(GRID.NPoints)
            }
    
    prop_fill = {'dens_Hnuc': 0.0,
                 'temp_gas': 2.725,
                 'temp_dust': 2.725,
                 'chem_HI' : 1e-2,
                 'chem_CO': 1e-12,
                 'vel_x': 1e6,
                 'vel_y': 1e6,
                 'vel_z': 1e6,
                 'gtdratio': 100,
                 }
    
    return prop, prop_fill



#******************
#MERGING TWIN CELLS
#******************
random_choice = False #If True, takes only a portion (npoints below) of the whole set of grid points
d_obj_orig = deepcopy(d_obj) 
unique_cells = UniqueCells(data, header)
id_pos = unique_cells.mergemass()   # Indices of non-repeated cells.

if random_choice:
    npoints = 100000
    id_pos_0 = id_pos
    id_pos = np.random.choice(id_pos, size=npoints, replace=False) # Select npoints indices from id_pos without replacement.

#************
#GRID OBJECT 
#************
#pos should transfer to the physics scale
pos_max = np.max(pos, axis=0)
pos_min = np.min(pos, axis=0)
pos_mean = 0.5 * (pos_max + pos_min) * arepoLength  # Midpoint between the coordinate minima and maxima.

GRID = Struct( XYZ = np.array([ pos[:,i][id_pos] * arepoLength - pos_mean[i] for i in range(3)]) )  # Translate the coordinate frame.
GRID.rRTP = [np.linalg.norm(GRID.XYZ, axis=0)]    # Compute each column-vector norm along axis 0: distance from the origin.
GRID.NPoints = len(id_pos)
GRID.XYZ.shape    




In [ ]:
xt0=((GRID.XYZ)[0,:]/arepoLength)
yt0=((GRID.XYZ)[1,:]/arepoLength)
zt0=((GRID.XYZ)[2,:]/arepoLength)
plt.hist2d(xt0,zt0,bins=100)


# dummy

In [ ]:
#***************************
#FILLING EMPTY SPACES + LIME
#***************************
prop, prop_fill = get_prop_dicts(id_pos)   
fill = fillgrid.Random(GRID)     
fill_rand = fill.by_mass(d_obj.mass[id_pos], prop,
                         prop_fill = prop_fill,
                         mass_fraction = 0.8, 
                         n_dummy = GRID.NPoints*0.1,
                         r_max = limit*arepoLength,
                         r_steps = 50000)


points = np.array(GRID.XYZ).T 
print ("limit: ", limit)
print ("max_len: ", l_max)
print ("creating delaunay")



xt1=(points[:,0]/arepoLength)
yt1=(points[:,1]/arepoLength)
zt1=(points[:,2]/arepoLength)
plt.hist2d(xt1,yt1,bins=100)

In [ ]:

#**************************
#DELAUNAY AND VORONOI
#**************************
print('Begin:')
tri = Delaunay(points)
tets = tri.points[tri.simplices]
#vol = tetrahedron_volume(tets[:, 0], tets[:, 1], 
#                         tets[:, 2], tets[:, 3])

vor = Voronoi(points)

print ("finding all neigbors")
print ((vor,points/arepoLength)[1][:,0],np.max((vor,points/arepoLength)[1][:,0]))

neighbors = find_neighbors(tri)
sys.stdout.write(CLR_LINE)

print ("creating convex hull")
convex_hull = create_convex_hull_list(tri.convex_hull)
sys.stdout.write(CLR_LINE)
tri.close()

print ("length of convex hull: ", len(convex_hull))
print ("Delaunay: done")

data_len = len(data_ids)
l_max=l_max*arepoLength

file = open(output_file, "wb")

file.write(struct.pack("H", grid_id))
file.write(struct.pack("H", data_len))

for d_ids in data_ids:
    file.write(struct.pack("H", d_ids))

file.write(struct.pack("d", GRID.NPoints))
file.write(struct.pack("d", l_max))

pos_counter = 0
volume = np.zeros(GRID.NPoints)+1.0
volume2 = np.zeros(GRID.NPoints)+1.0
dv_turb = np.zeros(GRID.NPoints)+1.0
dv_max = np.zeros(GRID.NPoints)+1.0
dv_combine = np.zeros(GRID.NPoints)+1.0

max_d = np.zeros(GRID.NPoints)+1.0
mean_d = np.zeros(GRID.NPoints)+1.0

for i in range(GRID.NPoints):
    x,y,z = points[i]
    dens = prop['dens_Hnuc'][i] #H nucleons number density #in 1/m3
    Td = prop['temp_dust'][i] #in K
    Tg = prop['temp_gas'][i]  #in K
    velx = prop['vel_x'][i]
    vely = prop['vel_y'][i]
    velz = prop['vel_z'][i]
    #rCO = prop['chem_CO'][i] #This one is just the abundance... POLARIS extracts the mass info from the 13co.dat file
    rHI = prop['chem_HI'][i]
    
    vor_region_i = np.array(vor.regions[vor.point_region[i]]) #Indices of vertices forming region i.
    vor_region_valid = vor_region_i[vor_region_i!=-1] #-1 indicates vertex outside the Voronoi diagram.
    verts_region_valid = vor.vertices[vor_region_valid]
    delau_i = Delaunay(verts_region_valid)
    tets_i = delau_i.points[delau_i.simplices]
    vol2 = tetrahedron_volume(tets_i[:, 0], tets_i[:, 1], 
                              tets_i[:, 2], tets_i[:, 3]).sum()

    max_di, mean_di = tetrahedron_volume_and_distances(tets_i[:, 0], tets_i[:, 1], 
                                                        tets_i[:, 2], tets_i[:, 3])
    #vol = rel_volume*data['mass'][i]/data['rho'][i]*arepoVol #in m3
    #volume[pos_counter] = vol
    volume2[i] = vol2
    max_d[i] = max_di
    mean_d[i] =mean_di

    #Bfield is in units of Gauss
    #magx = data['Bfield'][i][0]
    #magy = data['Bfield'][i][1]
    #magz = data['Bfield'][i][2]

    file.write(struct.pack("f", x))
    file.write(struct.pack("f", y))
    file.write(struct.pack("f", z))

    file.write(struct.pack("d", vol2))

    file.write(struct.pack("f", dens))
    file.write(struct.pack("f", Td))
    file.write(struct.pack("f", Tg))

    #file.write(struct.pack("f", magx))
    #file.write(struct.pack("f", magy))
    #file.write(struct.pack("f", magz))

    file.write(struct.pack("f", velx))
    file.write(struct.pack("f", vely))
    file.write(struct.pack("f", velz))
    
    file.write(struct.pack("f", rHI))
    #file.write(struct.pack("f", rCO))
    
    
    ##########################################################################################
    #non-thermal turbulence velocity dv=0.87L^0.65
    #thermal sound speed (5/3ktmH)**0.5
    #choose the dv
    ####################################
    
    file.write(struct.pack("f", prop['doppler'][i]))
    
    #------##################################################

    Li = vol2**(1/3)   #size unit pc,arepoLength is 100 pc,volume2 is m3
    dv_turb_i = 1000*0.87*(Li/arepoLength*100)**0.65 #unit m/s
    dv_turb[i] = dv_turb_i
    
    
    #file.write(struct.pack("f", dv_turb_i))

    
    #------##################################################
    #dv_combine[i] = (dv_turb_i**2+prop['doppler'][i]**2)**0.5
    
    #if dv_turb_i>prop['doppler'][i]:
        #dv_max[i] = dv_turb_i
    #else:
        #dv_max[i] = prop['doppler'][i]
    
    #file.write(struct.pack("f", dv_combine[i]))
    #file.write(struct.pack("f", dv_max[i]))
    #-------################################################
    
    ######################################
    ##########################################################################################
    
    p_list = list(neighbors[pos_counter])
    sign = int(is_in_hull(convex_hull, pos_counter))
    neigbors = int(len(p_list))
    nr_of_neigbors = sign * neigbors

    file.write(struct.pack("i", int(nr_of_neigbors)))

    
    for j in range(0, int(abs(nr_of_neigbors))):
        tmp_n=int(p_list[j])
        file.write(struct.pack("i", tmp_n))

    pos_counter+=1
    if pos_counter % 10000 == 0:
        sys.stdout.write('Writing voronoi grid: ' + str(100.0 * pos_counter / GRID.NPoints) + '%    \r')
        sys.stdout.flush()

sys.stdout.write(CLR_LINE)
file.close()
print ("DONE")






# save

In [ ]:
def generate_polaris_cmd(input_dat, output_path):
    content = f"""
<common>
    <dust_component> "/home/guowh/Downloads/POLARIS-master-basic/input/dust/silicate.nk" "plaw" 1.0  -3.5 5e-09 1.0e-06 
    <axis1>    0    1    0
    <axis2>    1    0    0
    <mu>            1.00794
    <nr_threads>        -1
</common>

<task> 
    <gas_species> "/home/guowh/Downloads/POLARIS-master-basic/input/gas/h1.dat" 1 -1
    <detector_line nr_pixel = "302" vel_channels = "100"> 1      1      1      100000.     0.0     90.0     1.234e+20
    <vel_maps> 1
    <nr_threads>        -1
    <cmd>            CMD_LINE_EMISSION
    <path_grid>        "{input_dat}"  #the <path_grid_cgs> command considers 1e3 to convert num densities, which is wrong
    <path_out>        "./{output_path}/resulthi90/"
    <max_subpixel_lvl>    2
    <mass_fraction>        0.01 #dust to gas mass ratio

    #<write_inp_midplanes> 512
</task>
    """
    with open(f"{label}.cmd", "w") as file:
        file.write(content)

# Example call.
generate_polaris_cmd(output_file, label)


In [ ]:
prop['pos']=((points + pos_mean)/(arepoLength/100)) - 50000
prop['volume']=(volume2/(arepoLength/100)**3)
prop['dv_combine']=dv_combine
prop['dv_max']=dv_max
prop['dv_nontherm']=dv_turb/1000
prop['dv_therm']=prop['doppler']/1000
np.save(output_file+'.npy', prop) 

In [ ]:
fig_size = plt.rcParams["figure.figsize"]
fig_size[0]=10
fig_size[1]=10
plt.rcParams["figure.figsize"] = fig_size

ulength = 3.0856e18  #cm
umass = 1.991e33  #g
uvel = 1.0e5  #cm/s

utime = ulength/uvel
udensity = umass/ulength/ulength/ulength
uenergy= umass*uvel*uvel
ucolumn = umass/ulength/ulength
umag = umass**0.5 / ulength**1.5 * uvel

uMyr=utime/(60.*60.*24.*365.25*1.e6)

uparsec=ulength/3.0856e18
rho = prop['dens_Hnuc']
rho_cgs = rho * udensity
#energy_per_unit_mass = data['u_therm']*uenergy/umass
xCO = prop['chem_CO']
xH2 = prop['chem_H2']
xHp = prop['chem_Hplus']
xHI = prop['chem_HI']

xHe=0.1
mp = 1.6726231e-24
kb = 1.3806485e-16
#xHI = 1 - xHp -2*xH2
yn = rho_cgs/((1. + 4.0 * xHe) * mp) #nHtot
nHp = xHp*yn
nH2 = xH2*yn
nCO = xCO*yn

nHI = xHI*yn
x = xt1.reshape(xt1.size)
y = yt1.reshape(yt1.size)
z = zt1.reshape(zt1.size)
quantity = nHI
dynamical_range = 10
quantity = quantity.reshape(quantity.size)
extent = [x.min(),x.max(),y.min(),y.max()]
nx = 100
ny = 100
w = quantity
datacube = np.histogram2d(x, z, weights=w, bins = (nx, ny))[0]
#levels = np.logspace(np.log10(datacube.max())-dynamical_range, np.log10(datacube.max()), 256)
#norm = mc.BoundaryNorm(levels,256)
dl=2*limit*100*sfu.pc

datacube = (np.rot90(datacube))
datacube=datacube*dl/10000
levels = np.logspace(np.log10(datacube.max())-dynamical_range, np.log10(datacube.max()), 256)
norm = mc.BoundaryNorm(levels,256)
deltai = 2
#extent=[-1500/deltai,1500/deltai,-1500/deltai,1500/deltai]

fig, ax = plt.subplots()
im=ax.imshow(datacube, cmap='Blues',extent=extent, norm=norm)

ax.set_title('Arepo CO hist2d image',fontsize=14)
ax.set_xlabel('x [pc]',fontsize=18)
ax.set_ylabel('y [pc]',fontsize=18)
ax.set_aspect(1)
ax.grid = True
#cbar_ax = fig.add_axes([0.9, 0.124, 0.02, 0.755])
#plt.colorbar(im,cax=cbar_ax)
#plt.xlim(-250,250)
#plt.ylim(-250,250)

plt.show()